# Mental Rotation — Prompt Fix Evaluation

Standalone notebook to test prompt strategies for the mental-rotation benchmark.
Goal: **get above 50% chance level** (binary A/B choice, 83 trials).

### Prior results (Qwen 0.8B, phases 0-5)
All phase combinations plateau at ~59% — not statistically significant.
With 83 trials at chance=0.5, a one-tailed binomial test requires **≥50 correct (60.2%)** for p < 0.05.

### New strategies tested here

| ID | Strategy | Hypothesis |
|---|---|---|
| S0 | Baseline (manifest prompt) | Reproduction of existing behaviour |
| S1 | Composite grid image | Single labeled image avoids multi-image confusion |
| S2 | Elimination framing | "Find the mirror → pick the other" reverses cognitive load |
| S3 | Describe-then-decide | Verbal feature anchoring before comparison |
| S4 | Pairwise verification | Isolate each option into a separate yes/no call |
| S5 | Mirror-only sanity | Skip the rotated reference — can VLMs detect mirrors at all? |
| GPT | GPT ceiling | Establishes whether the task is solvable by any model |

In [ ]:
from __future__ import annotations

import base64
import csv
import json
import mimetypes
import os
import random
import re
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data").is_dir() and (p / "src").is_dir()),
    Path.cwd().parent.parent,
)

LABELS = ["A", "B"]
CHANCE = 0.5
DATA_ROOT = ROOT / "data"
MANIFEST_PATH = DATA_ROOT / "assets" / "manifest.csv"
RESULTS_DIR = ROOT / "results" / "prompt_optimization" / "mental-rotation" / "fixes"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

## 1. Load manifest and resolve images

In [ ]:
def _detect_image_dir() -> Path:
    """Find the mental-rotation image directory under data/assets/<version>/visual/."""
    assets = DATA_ROOT / "assets"
    for child in sorted(assets.iterdir(), reverse=True):
        candidate = child / "visual" / "mental-rotation"
        if candidate.is_dir() and any(candidate.iterdir()):
            return candidate
    raise FileNotFoundError(
        "No mental-rotation images found. Run:\n"
        "  python scripts/data_prep/download_levante_assets.py --task mental-rotation"
    )


def _build_image_index(directory: Path) -> dict[str, Path]:
    index: dict[str, Path] = {}
    if not directory.is_dir():
        return index
    for path in directory.iterdir():
        if path.is_file():
            index[path.stem] = path
            index[path.name] = path
    return index


def _extract_angle(item_uid: str) -> Optional[int]:
    """Extract rotation angle from item_uid like 'mrot_2d_rabbit_080_Rp-080'."""
    m = re.search(r"_(\d{3})_", item_uid)
    if m:
        return int(m.group(1))
    if item_uid.endswith("-000") or "_000" in item_uid or item_uid.startswith(("Rn-000", "Rp-000")):
        return 0
    return None


IMAGE_DIR = _detect_image_dir()
IMAGE_INDEX = _build_image_index(IMAGE_DIR)
print(f"Image dir: {IMAGE_DIR} ({len(IMAGE_INDEX) // 2} images)")

In [ ]:
def load_trials() -> list[dict]:
    """Load mental-rotation trials from manifest."""
    rows = []
    with open(MANIFEST_PATH, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["task"] == "mental-rotation":
                rows.append(row)

    trials = []
    skipped = 0
    for row in rows:
        answer = str(row["answer"]).strip()
        alternatives = [a.strip() for a in row["response_alternatives"].split(",") if a.strip()]
        all_options = [answer] + alternatives

        rng = random.Random(row["item_uid"])
        rng.shuffle(all_options)
        correct_idx = all_options.index(answer)
        correct_label = LABELS[correct_idx] if correct_idx < len(LABELS) else "?"

        option_image_paths = []
        missing = False
        for option in all_options:
            path = IMAGE_INDEX.get(option.strip())
            if path is None:
                missing = True
                break
            option_image_paths.append(str(path))

        prompt_image_stem = str(row.get("prompt_image", "")).strip()
        context_image_paths = []
        if prompt_image_stem and prompt_image_stem not in {"NA", "nan", "TODO", ""}:
            path = IMAGE_INDEX.get(prompt_image_stem)
            if path is None:
                missing = True
            else:
                context_image_paths.append(str(path))

        if missing:
            skipped += 1
            continue

        trials.append({
            "item_uid": row["item_uid"],
            "trial_type": str(row.get("trial_type", "")).strip(),
            "angle": _extract_angle(row["item_uid"]),
            "full_prompt": row.get("full_prompt", ""),
            "options": all_options,
            "option_image_paths": option_image_paths,
            "context_image_paths": context_image_paths,
            "correct_label": correct_label,
        })

    if skipped:
        print(f"Skipped {skipped} trials with missing images")
    return trials


TRIALS = load_trials()
trial_types = sorted(set(t["trial_type"] for t in TRIALS))
angles = sorted(set(t["angle"] for t in TRIALS if t["angle"] is not None))
print(f"Loaded {len(TRIALS)} trials")
print(f"  Trial types: {trial_types}  (2 = 2D silhouettes, 3 = 3D blocks)")
print(f"  Angles: {angles}")

## 2. Visualise sample trials

In [ ]:
from PIL import Image


def show_trial(trial: dict, title: str = ""):
    n_images = len(trial["context_image_paths"]) + len(trial["option_image_paths"])
    fig, axes = plt.subplots(1, n_images, figsize=(4 * n_images, 4))
    if n_images == 1:
        axes = [axes]
    idx = 0
    for p in trial["context_image_paths"]:
        axes[idx].imshow(Image.open(p))
        axes[idx].set_title("Reference")
        axes[idx].axis("off")
        idx += 1
    for i, p in enumerate(trial["option_image_paths"]):
        label = LABELS[i]
        is_correct = label == trial["correct_label"]
        axes[idx].imshow(Image.open(p))
        axes[idx].set_title(f"Option {label}" + (" ✓" if is_correct else ""))
        axes[idx].axis("off")
        idx += 1
    suptitle = title or f"{trial['item_uid']}  (type {trial['trial_type']}, {trial['angle']}°)"
    fig.suptitle(suptitle, fontsize=11)
    plt.tight_layout()
    plt.show()


# Show one 2D and one 3D example
for tt, label in [("2", "2D silhouette"), ("3", "3D block")]:
    subset = [t for t in TRIALS if t["trial_type"] == tt]
    if subset:
        show_trial(subset[len(subset) // 2], title=f"Sample {label}")

## 3. Model backends

Two backends: **local HuggingFace** (Qwen, InternVL, etc.) and **OpenAI API** (GPT-5.x).
Set `USE_GPT = True` and export `OPENAI_API_KEY` to enable the GPT ceiling test.

In [ ]:
# ── Local HF backend ─────────────────────────────────────────────────────

_hf_cache: dict = {}


def load_hf_model(model_id: str):
    """Load a HuggingFace VLM. Returns (model, processor, dtype, device)."""
    if model_id in _hf_cache:
        return _hf_cache[model_id]
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText

    device = "cuda" if torch.cuda.is_available() else (
        "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"
    )
    dtype = torch.bfloat16
    processor = AutoProcessor.from_pretrained(model_id, padding_side="left")
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=dtype, attn_implementation="sdpa",
    ).to(device)
    model.eval()
    result = (model, processor, dtype, device)
    _hf_cache[model_id] = result
    return result


def generate_hf(
    model, processor, dtype, device,
    prompt_text: str, image_paths: list[str],
    max_new_tokens: int = 128,
    system_prompt: str | None = None,
) -> str:
    import torch

    pil_images = [Image.open(p).convert("RGB") for p in image_paths]

    # Interleave images at <imageN> tags
    content: list[dict] = []
    if pil_images and re.search(r"<image\d+>", prompt_text):
        parts = re.split(r"(<image\d+>)", prompt_text)
        for part in parts:
            m = re.match(r"<image(\d+)>", part)
            if m:
                idx = int(m.group(1))
                if idx < len(pil_images):
                    content.append({"type": "image", "image": pil_images[idx]})
            elif part.strip():
                content.append({"type": "text", "text": part.strip()})
    else:
        for img in pil_images:
            content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": prompt_text})

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": content})

    try:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    inputs = processor(
        text=[text],
        images=pil_images if pil_images else None,
        return_tensors="pt",
        padding=True,
    ).to(device)

    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, do_sample=False, max_new_tokens=max_new_tokens,
        )

    generated_ids = output_ids[:, input_len:]
    raw = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip() or raw

In [ ]:
# ── GPT API backend ──────────────────────────────────────────────────────

def generate_gpt(
    prompt_text: str, image_paths: list[str],
    model_name: str = "gpt-5.3",
    max_new_tokens: int = 256,
    system_prompt: str | None = None,
    reasoning_effort: str = "low",
) -> str:
    """Generate via the OpenAI Responses API. Requires OPENAI_API_KEY."""
    import requests

    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("Set OPENAI_API_KEY to use GPT models.")

    def _img_part(path_str):
        raw = Path(path_str).read_bytes()
        mime = mimetypes.guess_type(path_str)[0] or "image/png"
        b64 = base64.b64encode(raw).decode("ascii")
        return {"type": "input_image", "image_url": f"data:{mime};base64,{b64}"}

    content: list[dict] = []
    if image_paths and re.search(r"<image\d+>", prompt_text):
        chunks = re.split(r"(<image\d+>)", prompt_text)
        for chunk in chunks:
            m = re.match(r"<image(\d+)>", chunk)
            if m:
                idx = int(m.group(1))
                if 0 <= idx < len(image_paths):
                    content.append(_img_part(image_paths[idx]))
            elif chunk.strip():
                content.append({"type": "input_text", "text": chunk.strip()})
    else:
        for p in image_paths:
            content.append(_img_part(p))
        if prompt_text.strip():
            content.append({"type": "input_text", "text": prompt_text.strip()})

    payload: dict = {
        "model": model_name,
        "input": [{"role": "user", "content": content}],
        "max_output_tokens": max(256, max_new_tokens),
    }
    if reasoning_effort:
        payload["reasoning"] = {"effort": reasoning_effort}

    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    resp = requests.post(
        "https://api.openai.com/v1/responses",
        headers=headers, json=payload, timeout=120,
    )
    if resp.status_code != 200:
        raise RuntimeError(f"OpenAI error {resp.status_code}: {resp.text[:500]}")

    data = resp.json()
    if isinstance(data.get("output_text"), str) and data["output_text"]:
        return data["output_text"].strip()
    chunks_out: list[str] = []
    for item in data.get("output", []) or []:
        for part in item.get("content", []) or []:
            if part.get("type") in {"output_text", "text"}:
                txt = part.get("text")
                if isinstance(txt, str) and txt.strip():
                    chunks_out.append(txt.strip())
    return "\n".join(chunks_out).strip()

## 4. Answer parsing

In [ ]:
def parse_answer(text: str, option_labels: list[str] = LABELS) -> Optional[str]:
    """Extract a label from model output. Returns None if unparseable."""
    text = text.strip()
    labels_upper = [la.upper() for la in option_labels]

    # JSON
    try:
        parsed = json.loads(text)
        ans = parsed.get("answer", "").strip().upper()
        if ans in labels_upper:
            return ans
    except (json.JSONDecodeError, AttributeError):
        pass

    # Embedded JSON
    m = re.search(r'"answer"\s*:\s*"?([A-Z])\b', text, re.IGNORECASE)
    if m and m.group(1).upper() in labels_upper:
        return m.group(1).upper()

    # Natural-language patterns
    for pat in (
        r"(?:the\s+)?(?:correct\s+)?answer\s+is\s+([A-B])\b",
        r"option\s+([A-B])\b",
        r"(?:choose|select|pick)\s+([A-B])\b",
        r"\(([A-B])\)",
    ):
        m = re.search(pat, text, re.IGNORECASE)
        if m and m.group(1).upper() in labels_upper:
            return m.group(1).upper()

    # Exact single label
    if text.upper() in labels_upper:
        return text.upper()

    # Starts with label + delimiter
    for label in option_labels:
        if text.upper().startswith(label.upper()):
            rest = text[len(label):]
            if not rest or rest[0] in " .),:;\n":
                return label.upper()

    # Reverse sentence scan — last mention wins
    for sentence in reversed(re.split(r"[.!?\n]", text)):
        for label in option_labels:
            if re.search(rf"\b{re.escape(label)}\b", sentence, re.IGNORECASE):
                return label.upper()

    return None


def parse_same_mirror(text: str) -> Optional[str]:
    """For S4 pairwise: extract SAME or MIRROR from model output."""
    text_up = text.strip().upper()
    has_same = bool(re.search(r"\bSAME\b", text_up))
    has_mirror = bool(re.search(r"\b(?:MIRROR|FLIP|REFLECT)", text_up))
    if has_same and not has_mirror:
        return "SAME"
    if has_mirror and not has_same:
        return "MIRROR"
    # Ambiguous — check last sentence
    for sentence in reversed(re.split(r"[.!?\n]", text_up)):
        if re.search(r"\bSAME\b", sentence):
            return "SAME"
        if re.search(r"\b(?:MIRROR|FLIP|REFLECT)", sentence):
            return "MIRROR"
    return None

## 5. Prompt strategies

In [ ]:
# ── S0: Baseline (manifest prompt) ──────────────────────────────────────

def prompt_s0_baseline(trial: dict) -> tuple[str, list[str]]:
    prompt = trial["full_prompt"]
    if "<prompt_image>" in prompt:
        prompt = prompt.replace("<prompt_image>", "<image0>")
    image_paths = trial["context_image_paths"] + trial["option_image_paths"]
    return prompt, image_paths


# ── S1: Composite grid image ────────────────────────────────────────────

def _make_composite(trial: dict, ref_size: int = 256, opt_size: int = 200, padding: int = 20) -> Image.Image:
    """Stitch reference + options into a single labeled canvas."""
    from PIL import ImageDraw

    ref_img = Image.open(trial["context_image_paths"][0]).convert("RGB").resize((ref_size, ref_size))
    opt_a = Image.open(trial["option_image_paths"][0]).convert("RGB").resize((opt_size, opt_size))
    opt_b = Image.open(trial["option_image_paths"][1]).convert("RGB").resize((opt_size, opt_size))

    total_w = max(ref_size, opt_size * 2 + padding) + 2 * padding
    total_h = ref_size + opt_size + 5 * padding + 30
    canvas = Image.new("RGB", (total_w, total_h), (255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    rx = (total_w - ref_size) // 2
    ry = padding + 20
    draw.text((total_w // 2 - 40, padding), "REFERENCE", fill="black")
    canvas.paste(ref_img, (rx, ry))

    oy = ry + ref_size + 2 * padding + 10
    ox_a = (total_w - opt_size * 2 - padding) // 2
    ox_b = ox_a + opt_size + padding
    draw.text((ox_a + opt_size // 2 - 5, oy - 16), "A", fill="black")
    canvas.paste(opt_a, (ox_a, oy))
    draw.text((ox_b + opt_size // 2 - 5, oy - 16), "B", fill="black")
    canvas.paste(opt_b, (ox_b, oy))

    return canvas


def prompt_s1_composite(trial: dict) -> tuple[str, list[str]]:
    if not trial["context_image_paths"]:
        return prompt_s0_baseline(trial)
    composite = _make_composite(trial)
    tmp_path = RESULTS_DIR / "composites" / f"{trial['item_uid']}.png"
    tmp_path.parent.mkdir(parents=True, exist_ok=True)
    composite.save(tmp_path)
    prompt = (
        "This image shows a REFERENCE shape at the top and two options (A, B) below.\n"
        "One option is the same shape rotated to a different angle. "
        "The other is its mirror image (horizontally flipped).\n"
        "Which option matches the reference — just rotated, NOT flipped?\n\n"
        "Answer with one letter: A or B."
    )
    return prompt, [str(tmp_path)]


# ── S2: Elimination framing ─────────────────────────────────────────────

def prompt_s2_elimination(trial: dict) -> tuple[str, list[str]]:
    prompt = (
        "Reference image:\n<image0>\n\n"
        "Below are two options. One is the SAME shape rotated to a different angle. "
        "The other is a MIRROR IMAGE (horizontally flipped).\n\n"
        "First, identify which option looks like a mirror/flip of the reference.\n"
        "Then pick the OTHER option — the one that is simply rotated.\n\n"
        "A: <image1>\nB: <image2>\n\n"
        "Answer with one letter: A or B."
    )
    image_paths = trial["context_image_paths"] + trial["option_image_paths"]
    return prompt, image_paths


# ── S3: Describe-then-decide ────────────────────────────────────────────

def prompt_s3_describe(trial: dict) -> tuple[str, list[str]]:
    prompt = (
        "Reference image:\n<image0>\n\n"
        "Option A: <image1>\nOption B: <image2>\n\n"
        "Task: One option is the same shape as the reference, just rotated. "
        "The other is its mirror image.\n\n"
        "Step 1: In the reference, identify one asymmetric feature "
        "(e.g. which direction does a protruding part point — left or right?).\n"
        "Step 2: Check option A — does that feature point the same way "
        "(accounting for rotation) or is it flipped?\n"
        "Step 3: Check option B the same way.\n"
        "Step 4: The option where the feature matches (not flipped) is the rotation.\n\n"
        "State your final answer as a single letter: A or B."
    )
    image_paths = trial["context_image_paths"] + trial["option_image_paths"]
    return prompt, image_paths


# ── S4: Pairwise verification (two calls per trial) ─────────────────────

def prompt_s4_pairwise(trial: dict, option_idx: int) -> tuple[str, list[str]]:
    """Ask about a single option: is it SAME (rotated) or MIRROR (flipped)?"""
    prompt = (
        "Reference image:\n<image0>\n\n"
        "Candidate image:\n<image1>\n\n"
        "Is the candidate the SAME shape as the reference, just rotated to a "
        "different angle? Or is it a MIRROR image (horizontally flipped)?\n\n"
        "Answer with one word: SAME or MIRROR."
    )
    image_paths = trial["context_image_paths"] + [trial["option_image_paths"][option_idx]]
    return prompt, image_paths


# ── S5: Mirror-only sanity check ────────────────────────────────────────

def prompt_s5_mirror_only(trial: dict) -> tuple[str, list[str]]:
    """Show just the two 0° options (no rotated reference). Tests basic mirror detection."""
    prompt = (
        "Below are two images of similar shapes. "
        "One is the original and the other is its mirror image (horizontally flipped).\n\n"
        "A: <image0>\nB: <image1>\n\n"
        "Which one is the original (not flipped)? If you can't tell, just guess.\n\n"
        "Answer with one letter: A or B."
    )
    return prompt, trial["option_image_paths"]


STRATEGIES = {
    "S0_baseline":    prompt_s0_baseline,
    "S1_composite":   prompt_s1_composite,
    "S2_elimination": prompt_s2_elimination,
    "S3_describe":    prompt_s3_describe,
    "S5_mirror_only": prompt_s5_mirror_only,
}

print("Strategies:", list(STRATEGIES.keys()))
print("S4 (pairwise) handled separately — requires two calls per trial.")

## 6. Evaluation loop

In [ ]:
def run_strategy(
    strategy_name: str,
    prompt_fn,
    trials: list[dict],
    generate_fn,
    max_tokens: int = 128,
    system_prompt: str | None = None,
) -> list[dict]:
    """Run a prompt strategy across all trials."""
    from tqdm.notebook import tqdm

    results = []
    for trial in tqdm(trials, desc=strategy_name):
        prompt, image_paths = prompt_fn(trial)
        try:
            response = generate_fn(
                prompt, image_paths,
                max_new_tokens=max_tokens,
                system_prompt=system_prompt,
            )
        except Exception as e:
            response = f"ERROR: {e}"

        predicted = parse_answer(response)
        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "raw_response": response,
            "strategy": strategy_name,
        })
    return results


def run_pairwise(
    trials: list[dict],
    generate_fn,
    max_tokens: int = 128,
    system_prompt: str | None = None,
) -> list[dict]:
    """S4: two calls per trial — ask SAME/MIRROR for each option independently."""
    from tqdm.notebook import tqdm

    results = []
    for trial in tqdm(trials, desc="S4_pairwise"):
        prompt_a, paths_a = prompt_s4_pairwise(trial, option_idx=0)
        prompt_b, paths_b = prompt_s4_pairwise(trial, option_idx=1)
        try:
            resp_a = generate_fn(prompt_a, paths_a, max_new_tokens=max_tokens,
                                 system_prompt=system_prompt)
            resp_b = generate_fn(prompt_b, paths_b, max_new_tokens=max_tokens,
                                 system_prompt=system_prompt)
        except Exception as e:
            resp_a = resp_b = f"ERROR: {e}"

        verdict_a = parse_same_mirror(resp_a)
        verdict_b = parse_same_mirror(resp_b)

        if verdict_a == "SAME" and verdict_b != "SAME":
            predicted = "A"
        elif verdict_b == "SAME" and verdict_a != "SAME":
            predicted = "B"
        else:
            predicted = None  # ambiguous

        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "raw_response": f"A:{resp_a} | B:{resp_b}",
            "strategy": "S4_pairwise",
        })
    return results

## 7. Configuration

In [ ]:
HF_MODEL_ID = "Qwen/Qwen3.5-0.8B"  # Change to test other models

USE_GPT = bool(os.environ.get("OPENAI_API_KEY"))
GPT_MODEL = "gpt-5.3"

LIMIT = None  # Set to e.g. 10 for a quick smoke test

SYSTEM_PROMPT = (
    "You are a visual reasoning assistant. "
    "Answer with only a single letter: A or B. Do not explain."
)

trials_to_run = TRIALS[:LIMIT] if LIMIT else TRIALS
print(f"Trials: {len(trials_to_run)}")
print(f"HF model: {HF_MODEL_ID}")
print(f"GPT enabled: {USE_GPT}" + (f" ({GPT_MODEL})" if USE_GPT else ""))

## 8. Run local HF model

In [ ]:
hf_model, hf_processor, hf_dtype, hf_device = load_hf_model(HF_MODEL_ID)
print(f"Loaded {HF_MODEL_ID} on {hf_device}")


def gen_hf(prompt, image_paths, max_new_tokens=128, system_prompt=None):
    return generate_hf(
        hf_model, hf_processor, hf_dtype, hf_device,
        prompt, image_paths, max_new_tokens,
        system_prompt=system_prompt or SYSTEM_PROMPT,
    )


all_results: dict[str, list[dict]] = {}

for name, fn in STRATEGIES.items():
    tok = 256 if "describe" in name else 128
    all_results[name] = run_strategy(
        name, fn, trials_to_run, gen_hf,
        max_tokens=tok, system_prompt=SYSTEM_PROMPT,
    )

all_results["S4_pairwise"] = run_pairwise(
    trials_to_run, gen_hf,
    max_tokens=128, system_prompt=SYSTEM_PROMPT,
)

print(f"\nLocal model runs complete ({len(all_results)} strategies).")

## 9. Run GPT ceiling test (optional)

In [ ]:
if USE_GPT:
    def gen_gpt(prompt, image_paths, max_new_tokens=256, system_prompt=None):
        return generate_gpt(
            prompt, image_paths,
            model_name=GPT_MODEL,
            max_new_tokens=max_new_tokens,
            reasoning_effort="low",
        )

    gpt_strategies = {
        "GPT_baseline": prompt_s0_baseline,
        "GPT_describe": prompt_s3_describe,
        "GPT_elimination": prompt_s2_elimination,
    }
    for name, fn in gpt_strategies.items():
        tok = 512 if "describe" in name else 256
        all_results[name] = run_strategy(
            name, fn, trials_to_run, gen_gpt, max_tokens=tok,
        )

    all_results["GPT_pairwise"] = run_pairwise(
        trials_to_run, gen_gpt, max_tokens=256,
    )
    print(f"GPT runs complete ({GPT_MODEL}).")
else:
    print("Skipping GPT — set OPENAI_API_KEY to enable.")

## 10. Summary results

In [ ]:
def _binom_p(correct: int, n: int) -> float:
    """One-tailed binomial test: H1 = accuracy > chance."""
    if n == 0:
        return 1.0
    return stats.binomtest(correct, n, CHANCE, alternative="greater").pvalue


summary_rows = []
for name in sorted(all_results):
    results = all_results[name]
    n = len(results)
    correct = sum(1 for r in results if r["is_correct"])
    parsed = sum(1 for r in results if r["predicted_label"] is not None)
    acc = correct / n if n else 0
    p_val = _binom_p(correct, n)
    summary_rows.append({
        "Strategy": name,
        "N": n,
        "Correct": correct,
        "Accuracy": acc,
        "Parse %": parsed / n if n else 0,
        "p-value": p_val,
        "Sig (p<.05)": "Yes" if p_val < 0.05 else "No",
    })

df_summary = pd.DataFrame(summary_rows).sort_values("Accuracy", ascending=False)
display(
    df_summary.style
    .format({"Accuracy": "{:.1%}", "Parse %": "{:.1%}", "p-value": "{:.4f}"})
    .applymap(
        lambda v: "background-color: #c6efce" if v == "Yes" else "",
        subset=["Sig (p<.05)"],
    )
    .hide(axis="index")
)

## 11. Breakdown: 2D silhouettes vs 3D blocks

In [ ]:
TYPE_LABELS = {"2": "2D silhouettes", "3": "3D blocks"}

breakdown_rows = []
for name in sorted(all_results):
    for tt in ("2", "3"):
        subset = [r for r in all_results[name] if r["trial_type"] == tt]
        if not subset:
            continue
        n = len(subset)
        correct = sum(1 for r in subset if r["is_correct"])
        acc = correct / n
        p_val = _binom_p(correct, n)
        breakdown_rows.append({
            "Strategy": name,
            "Type": TYPE_LABELS.get(tt, tt),
            "N": n,
            "Accuracy": acc,
            "p-value": p_val,
        })

df_bd = pd.DataFrame(breakdown_rows)
pivot = df_bd.pivot_table(
    index="Strategy", columns="Type", values="Accuracy", aggfunc="first",
)
display(
    pivot.style
    .format("{:.1%}")
    .background_gradient(cmap="RdYlGn", vmin=0.35, vmax=0.80)
)

## 12. Accuracy by rotation angle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (tt, tt_label) in zip(axes, [("2", "2D silhouettes"), ("3", "3D blocks")]):
    for name in sorted(all_results):
        results = all_results[name]
        subset = [r for r in results if r["trial_type"] == tt and r["angle"] is not None]
        if not subset:
            continue
        df_tmp = pd.DataFrame(subset)
        angle_acc = df_tmp.groupby("angle")["is_correct"].mean()
        ax.plot(angle_acc.index, angle_acc.values, "o-", label=name, alpha=0.7, markersize=4)
    ax.axhline(CHANCE, color="gray", linestyle=":", linewidth=1, label="chance")
    ax.set_xlabel("Rotation angle (degrees)")
    ax.set_ylabel("Accuracy")
    ax.set_title(tt_label)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=7, loc="lower left")

fig.suptitle("Accuracy by rotation angle and strategy", fontsize=13)
plt.tight_layout()
plt.show()

## 13. Error analysis

In [ ]:
best_name = df_summary.iloc[0]["Strategy"]
best_results = all_results[best_name]
wrong = [r for r in best_results if not r["is_correct"] and r["predicted_label"] is not None]
unparsed = [r for r in best_results if r["predicted_label"] is None]

print(f"Best strategy: {best_name}")
print(f"  Errors (parsed but wrong): {len(wrong)}")
print(f"  Unparsed: {len(unparsed)}")
print()

if wrong:
    print("Sample errors:")
    for r in wrong[:5]:
        trial = next(t for t in TRIALS if t["item_uid"] == r["item_uid"])
        print(f"  {r['item_uid']}  type={r['trial_type']}  angle={r['angle']}")
        print(f"    Predicted: {r['predicted_label']}  Correct: {r['correct_label']}")
        print(f"    Response: {r['raw_response'][:150]}")
        print()

if unparsed:
    print("Sample unparsed:")
    for r in unparsed[:3]:
        print(f"  {r['item_uid']}: {r['raw_response'][:150]}")
        print()

In [ ]:
# Visualise a few errors from the best strategy
for r in wrong[:3]:
    trial = next(t for t in TRIALS if t["item_uid"] == r["item_uid"])
    show_trial(
        trial,
        title=f"ERROR — predicted {r['predicted_label']}, correct {r['correct_label']} "
              f"({r['item_uid']}, {r['angle']}°)",
    )

## 14. Response bias check

If a model always picks A or B regardless of images, accuracy will be ~50% by luck.
A chi-squared test checks whether the distribution of predicted labels deviates from uniform.

In [ ]:
bias_rows = []
for name in sorted(all_results):
    results = all_results[name]
    parsed = [r for r in results if r["predicted_label"] is not None]
    if not parsed:
        continue
    counts = {"A": 0, "B": 0}
    for r in parsed:
        label = r["predicted_label"]
        if label in counts:
            counts[label] += 1
    n = len(parsed)
    expected = n / 2
    chi2 = sum((v - expected) ** 2 / expected for v in counts.values())
    p_val = 1 - stats.chi2.cdf(chi2, df=1)
    bias_rows.append({
        "Strategy": name,
        "N parsed": n,
        "A count": counts["A"],
        "B count": counts["B"],
        "A %": f"{counts['A'] / n:.0%}",
        "chi2 p": f"{p_val:.3f}",
        "Biased (p<.05)": "Yes" if p_val < 0.05 else "No",
    })

display(pd.DataFrame(bias_rows).style.hide(axis="index"))

## 15. Save results

In [ ]:
for name, results in all_results.items():
    path = RESULTS_DIR / f"{name}.csv"
    pd.DataFrame(results).to_csv(path, index=False)
    print(f"Saved {path.name} ({len(results)} rows)")

summary_path = RESULTS_DIR / "summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"\nSaved {summary_path.name}")

## 16. Conclusions

**Fill in after running:**

1. **GPT ceiling:** GPT-5.3 achieved ___% on baseline / ___% on describe-then-decide (p=___).
   - If at chance: task images may need redesign.
   - If above chance: smaller models need better prompting or fine-tuning.

2. **Best local strategy:** ___ achieved ___% (p=___).
   - Significantly above chance? [Yes/No]

3. **Composite grid (S1):** ___% — does single-image delivery help?

4. **Pairwise verification (S4):** ___% — does isolating comparisons help?

5. **Mirror-only sanity (S5):** ___% — can VLMs detect mirror reflection at all?

6. **Response bias:** Is the model just always picking A (or B)?

### Recommendations for the benchmark

- [ ] If GPT ceiling < 60%: consider redesigning mental rotation stimuli
- [ ] If S1 (composite) >> S0 (baseline): switch to single-image delivery
- [ ] If S5 (mirror-only) is at chance: VLMs lack mirror detection — task may not be appropriate
- [ ] If a strategy reaches significance: update `manifest.csv` `full_prompt`
- [ ] If model is biased toward one label: ensure balanced option shuffling